# Constructing an ensemble of GEMs using medusa

In this script, we will take the curated pan-GEM of Aspergillus oryzae obtained in *curate_panAsp.mlx* and the table of orthologous gene assignments to reconstruct the individual models for the different Aspergillus strains as obtained from BPGA using the usearch algorithm. For each reaction in the pan-GEM, we here also obtain the corresponding pathway (subsystem) label from the KEGG database. Subsequently, we use the Pyhton library *Medusa* to reformat this list of individual models into a single ensemble model object, that we will use downstream for performing FBA simulations.

## Load libraries

In [1]:
import medusa
import cobra
import numpy
from pathlib import Path
from pickle import load
from cobra.io import read_sbml_model

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Load pan-Oryzae GEM

In [ ]:
data_dir = Path("../data/intermediate")
data_dir = data_dir.resolve()
model_path = data_dir / "pAo_draft.xml"

panOryzae = read_sbml_model(str(model_path.resolve()), skip_validation=True)

Set parameter Username
Set parameter LicenseID to value 2681871
Academic license - for non-commercial use only - expires 2026-06-24


https://identifiers.org/ec-code/ does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id


## Obtain KEGG pathway label for each reaction

In [3]:
import re
import requests

# Add subsystem information to the reactions that come from KEGG
def extract_reaction_id(input_string):
    match = re.search(r'R\d{5}', input_string)  # Pattern 'R' followed by 5 digits
    if match:
        return match.group(0)  # Return the matched string
    # No return if no match is found (implicitly returns None, but we won't include it in the list)

allRids = [rid.id for rid in panOryzae.reactions]
keggRids = [extract_reaction_id(rid) for rid in allRids if extract_reaction_id(rid) != None]

# Function to get all pathway names and IDs from KEGG
def get_all_pathways():
    url = 'http://rest.kegg.jp/list/pathway'
    response = requests.get(url)
    
    if response.status_code == 200:
        pathway_dict = {}
        # Split the response into lines and process each line
        for line in response.text.strip().split('\n'):
            # Extract pathway_id and pathway_name
            pathway_id, pathway_name = line.split('\t')
            # We are only interested in "map" pathways
            if pathway_id.startswith('map'):
                pathway_dict[pathway_id] = pathway_name.strip()
        return pathway_dict
    else:
        print("Failed to fetch pathways.")
        return {}

# Example usage
pathway_dict = get_all_pathways()
pathway_dict

# Function to get pathways for a given KEGG reaction ID and their names
def get_pathways_for_reaction(reaction_id):
    url = f'http://rest.kegg.jp/link/pathway/{reaction_id}'
    response = requests.get(url)

    if response.status_code == 200:
        if response.text.strip() != '':
            # Split the response into lines and filter for 'map' pathways
            pathways = [line.split('\t')[1].replace('path:', '') for line in response.text.strip().split('\n') if 'map' in line.split('\t')[1]]
            return pathways
        else:
            return []
    else:
        return []

# Get pathways and pathway names for the reaction
pathway_ids = get_pathways_for_reaction(keggRids[1])
print(f"Pathways for {keggRids[1]}: {pathway_ids}")

# Retrieve pathway names from the dictionary
pathway_names = [pathway_dict.get(pathway_id, "Unknown pathway") for pathway_id in pathway_ids]
pathway_names
results = []
for rid in keggRids:
    # Get pathways and pathway names for the reaction
    pathway_ids = get_pathways_for_reaction(rid)
    # Retrieve pathway names from the dictionary
    pathway_names = [pathway_dict.get(pathway_id, "Unknown pathway") for pathway_id in pathway_ids]
    print(f"Pathways for {rid}: {pathway_names}")
    results.append(pathway_names)


{'map01100': 'Metabolic pathways',
 'map01110': 'Biosynthesis of secondary metabolites',
 'map01120': 'Microbial metabolism in diverse environments',
 'map01200': 'Carbon metabolism',
 'map01210': '2-Oxocarboxylic acid metabolism',
 'map01212': 'Fatty acid metabolism',
 'map01230': 'Biosynthesis of amino acids',
 'map01232': 'Nucleotide metabolism',
 'map01250': 'Biosynthesis of nucleotide sugars',
 'map01240': 'Biosynthesis of cofactors',
 'map01220': 'Degradation of aromatic compounds',
 'map01310': 'Nitrogen cycle',
 'map01320': 'Sulfur cycle',
 'map00010': 'Glycolysis / Gluconeogenesis',
 'map00020': 'Citrate cycle (TCA cycle)',
 'map00030': 'Pentose phosphate pathway',
 'map00040': 'Pentose and glucuronate interconversions',
 'map00051': 'Fructose and mannose metabolism',
 'map00052': 'Galactose metabolism',
 'map00053': 'Ascorbate and aldarate metabolism',
 'map00500': 'Starch and sucrose metabolism',
 'map00620': 'Pyruvate metabolism',
 'map00630': 'Glyoxylate and dicarboxylate 

Pathways for R00014: ['map00010', 'map00020', 'map00620', 'map00785']


['Glycolysis / Gluconeogenesis',
 'Citrate cycle (TCA cycle)',
 'Pyruvate metabolism',
 'Lipoic acid metabolism']

Pathways for R00006: []
Pathways for R00014: ['Glycolysis / Gluconeogenesis', 'Citrate cycle (TCA cycle)', 'Pyruvate metabolism', 'Lipoic acid metabolism']
Pathways for R00025: ['Nitrogen metabolism', 'Metabolic pathways']
Pathways for R00026: []
Pathways for R00089: ['Purine metabolism', 'Metabolic pathways']
Pathways for R00177: ['Cysteine and methionine metabolism', 'One carbon pool by folate', 'Biosynthesis of various plant secondary metabolites', 'Metabolic pathways', 'Biosynthesis of secondary metabolites', 'Biosynthesis of amino acids', 'Biosynthesis of cofactors']
Pathways for R00179: ['Cysteine and methionine metabolism', 'Metabolic pathways', 'Biosynthesis of secondary metabolites']
Pathways for R00220: ['Glycine, serine and threonine metabolism', 'Metabolic pathways', 'Biosynthesis of secondary metabolites', 'Carbon metabolism', 'Biosynthesis of amino acids']
Pathways for R00226: ['Valine, leucine and isoleucine biosynthesis', 'Butanoate metabolism', 'C5-Branched dibasic aci

## Inspect the obtained pathway labels

In [4]:
from collections import Counter

# Step 1: Extract the first element of each array
first_elements = [arr[0] for arr in results if arr]
# Step 2: Count the frequency of each string
frequency = Counter(first_elements)
frequency

# Step 1: Extract the last element of each array
last_elements = [arr[-1] for arr in results if arr]
# Step 2: Count the frequency of each string
frequency = Counter(last_elements)
frequency

Counter({'Fatty acid biosynthesis': 38,
         'Metabolism of xenobiotics by cytochrome P450': 34,
         'Arginine and proline metabolism': 18,
         'Steroid hormone biosynthesis': 18,
         'Cysteine and methionine metabolism': 17,
         'Arachidonic acid metabolism': 15,
         'Valine, leucine and isoleucine degradation': 10,
         'Tyrosine metabolism': 10,
         'Glycine, serine and threonine metabolism': 9,
         'Starch and sucrose metabolism': 9,
         'Fatty acid degradation': 9,
         'Galactose metabolism': 8,
         'Cyanoamino acid metabolism': 8,
         'Aflatoxin biosynthesis': 8,
         'Tryptophan metabolism': 7,
         'Phenylpropanoid biosynthesis': 7,
         'Glycerophospholipid metabolism': 6,
         'Pentose and glucuronate interconversions': 6,
         'Steroid biosynthesis': 6,
         'Staurosporine biosynthesis': 6,
         'Biosynthesis of various other secondary metabolites': 6,
         'Purine metabolism': 5,


Counter({'Metabolic pathways': 149,
         'Biosynthesis of secondary metabolites': 73,
         'Fatty acid metabolism': 42,
         'Metabolism of xenobiotics by cytochrome P450': 34,
         'Microbial metabolism in diverse environments': 27,
         'Biosynthesis of cofactors': 20,
         'Biosynthesis of amino acids': 18,
         'Degradation of aromatic compounds': 7,
         'Steroid hormone biosynthesis': 7,
         'Nucleotide metabolism': 5,
         'Drug metabolism - other enzymes': 5,
         'Glycerophospholipid metabolism': 4,
         'Aminoacyl-tRNA biosynthesis': 3,
         '2-Oxocarboxylic acid metabolism': 3,
         'Alanine, aspartate and glutamate metabolism': 2,
         'Fatty acid degradation': 2,
         'Carbon metabolism': 2,
         'Retinol metabolism': 2,
         'Fatty acid biosynthesis': 2,
         'Lipoic acid metabolism': 1,
         'Phenylalanine metabolism': 1,
         'Tryptophan metabolism': 1,
         'Starch and sucrose meta

## Add the pathway label to the corresponding reaction

In [5]:
for rxn in panOryzae.reactions:
    match = re.search(r'R\d{5}', rxn.id)  # Pattern 'R' followed by 5 digits
    if match:
        index = keggRids.index(match.group(0))
        if len(results[index]) > 0:
            rxn.subsystem = results[index][0] # add them to the subsytem slot of the model

## Import table of orthologous gene clusters

In [6]:
# Import BPGA table 
import pandas as pd
BPGA = pd.read_csv('../data/genome/BPGA2ortho_GEM_custom_187_50.csv', delimiter=";", dtype=str)

# Retain columns only for oryzae isolates and the oryzae tamplate model (as a reference)
BPGA.drop(['fumigatus','niger'], axis=1, inplace=True)
BPGA.rename(columns={'oryzae':'template'}, inplace=True)

# Many genes in the BPGA table are not used by any of the oryzae isolates
# -> remove these
model_sub = panOryzae.copy()
all_genes = [gene.id for gene in model_sub.genes]
BPGA = BPGA[BPGA['cluster'].isin(all_genes)]

# Make separate df for the isolate info and the gene info
BPGA_gene = BPGA[['cluster','present_in_n_genomes','cluster_type_manual']]
BPGA_isolates = BPGA.drop(['cluster','present_in_n_genomes','cluster_type_manual'], axis=1, inplace=False)

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp4xt1aw9q.lp
Reading time = 0.05 seconds
: 1760 rows, 3892 columns, 15016 nonzeros


## Contruct a list of strain-specific models

Based on the pan-GEM and the table of orthologous gene clusters, we here contruct a list of strain-specific models. For each strain, this is achieved by taking the pan-GEM and pruning reactions catalyzed by gene (orthologous gene clusters) that are not observed in the specific strain.

Note: `remove_genes` removes genes as well as reactions catalyzed by these genes, but not metabolites. Indeed, subsequently running `prune_unused_reactions` does not alter the model. Removing unused metabolites can be achieved by running `prune_unused_metabolites`. In addition, it does not touch reactions without gene association, which is a desired property.

In [7]:
from cobra import manipulation

gemList = []
for i in range(len(BPGA_isolates.columns)):
    gem_i = panOryzae.copy()
    isolate_name = BPGA_isolates.columns[i]
    genesToBeRemoved = BPGA_gene.cluster.values[BPGA_isolates[isolate_name].isna().values]
    manipulation.remove_genes(gem_i,
                              genesToBeRemoved,
                              True)
    _, _, = manipulation.prune_unused_metabolites(gem_i)
    gem_i.id = isolate_name
    gemList.append(gem_i)

# Also include the pan-model for downstream use (e.g. to gapfill from)
panOryzae.id = 'Pan_oryzae'
gemList.append(panOryzae)

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmphq_th8yp.lp
Reading time = 0.05 seconds
: 1760 rows, 3892 columns, 15016 nonzeros


c:\Users\gilis\OneDrive - Chalmers\Desktop\postdoc\Aspergillus\panAsp-GEM\code\.venv\Lib\site-packages\cobra\core\group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpvx0pgy5m.lp
Reading time = 0.05 seconds
: 1760 rows, 2822 columns, 10028 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpq7tl4mp4.lp
Reading time = 0.10 seconds
: 1760 rows, 3892 columns, 15016 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp7jfwu_p1.lp
Reading time = 0.05 seconds
: 1760 rows, 3142 columns, 11522 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp6upqkrtz.lp
Reading time = 0.05 seconds
: 1760 rows, 3892 columns, 15016 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpj17dlc17.lp
Reading time = 0.11 seconds
: 1760 rows, 3244 columns, 11984 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpdqkkj_xc.lp
Reading time = 0.09 seconds
: 1760 rows, 3892 columns, 15016 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpkxv27_qc.lp
Reading time = 0.05 

## Wrangle the list of individual strain-specific models into a Medusa ensemble

In [8]:
from medusa.core import Ensemble
testEnsemble = Ensemble(list_of_models = gemList,
                        identifier = "testID",
                        name = "testName")

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpzos9m1mg.lp
Reading time = 0.08 seconds
: 1760 rows, 2822 columns, 10028 nonzeros


## Export objects

In [ ]:
import pickle
path = "../data/intermediate/"

# Export the Medusa ensemble as a pickle file 
pickle.dump(testEnsemble, open(path + "ensemble_187.pickle","wb"))

# Export the list of strain-specific GEMs as a pickle file 
pickle.dump(gemList, open(path + "strain-GEMs_automated_187.pickle","wb"))

# Export a shortlist of the strain-specific GEMs that will be used for FBA simulations as a pickle file
submembers = ['Pan_oryzae', 'template', 'Aspergillus_oryzae_RIB40_GCF_000184455.2', 'Aspergillus_oryzae_NRRL_2217',
              'Aspergillus_oryzae_NRRL_3483_GCA_034767915.1','Aspergillus_oryzae_NRRL_5589','Aspergillus_oryzae_NRRL_3488',
              'Aspergillus_oryzae_NRRL_5592_GCA_034767935.1','Aspergillus_oryzae_NRRL_35890_GCA_034767955.1','Aspergillus_oryzae_NRRL_471']

shortList = []
for mem in gemList:
    if mem.id in submembers:
        shortList.append(mem)

pickle.dump(shortList, open(path + "strain-GEMs_subset_10.pickle","wb"))

# Export the list of strain-specific GEMs as individual xml files
from cobra.io import write_sbml_model
for model in gemList:
    hlp = model.id.replace('.', '_')
    path = "../data/allModels/" + hlp + ".xml"
    write_sbml_model(model, path)